# Tokenization
자모 - 음절 - 단어 단위로 문장을 잘라서 ID로 표현  
(ID는 단순한 숫자, 의미 X)  

## vocabulary
ID와 자른 단어/음절의 사전  
단어 단위로 자를 경우 새로운 단어를 사전에 저장 못 할 수 있음. (= OOV(out of vocabulary))  
자모 단위로 자를 경우 의미 파악할 수 없음 (대신 OOV는 방지할 수 있음)  

## subword
자를 단위 결정 방식 (OOV 방지용)  
자주 등장하면 단어 (ex. 국가/유명인/자주 표현...)  
자주 등장 안 하면 음절로 자름 (ex. 외국어/특수 문자/이모티콘...)  

In [2]:
# 띄어쓰기 단위로 분리
input_text = "나는 최근 파리 여행을 다녀왔다."
input_text_list = input_text.split()
print("input_text_list: ", input_text_list)

# 토큰 -> 아이디 딕셔너리와 아이디 -> 토큰 딕셔너리 만들기
str2idx = {word:idx for idx, word in enumerate(input_text_list)}
idx2str = {idx: word for idx, word in enumerate(input_text_list)}
print("str2idx: ", str2idx)
print("idx2str: ", idx2str)

# 토큰을 토큰 아이디로 변환
input_ids = [str2idx[word] for word in input_text_list]
print("input_ids: ", input_ids)

input_text_list:  ['나는', '최근', '파리', '여행을', '다녀왔다.']
str2idx:  {'나는': 0, '최근': 1, '파리': 2, '여행을': 3, '다녀왔다.': 4}
idx2str:  {0: '나는', 1: '최근', 2: '파리', 3: '여행을', 4: '다녀왔다.'}
input_ids:  [0, 1, 2, 3, 4]


# Embedding
데이터에 의미를 담은 숫자 집합(Vector)로 표현

## Vector
2개 이상의 숫자 집합 (숫자로 의미를 가지려면 조합이 필요함)  
(ex. pyTorch의 nn.Embedding을 사용. dim을 통해 몇 개의 숫자 조합으로 표현할 지 결정)  
하지만 이렇게 나온 숫자는 의미 X (DL 모델이 생성하지 않았고, 학습하지 않음)  
(DL의 경우 학습을 하는 과정(토큰화: 자를 단위 결정(의미/특징 분석), 임베딩: 의미(vector) 부여)에서  
  더 좋은 임베딩 방식을 찾고 적용)

In [3]:
import sys
!"{sys.executable}" -m pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import torch
import torch.nn as nn

embedding_dim = 16
embed_layer = nn.Embedding(len(str2idx), embedding_dim)

input_embeddings = embed_layer(torch.tensor(input_ids))
input_embeddings = input_embeddings.unsqueeze(0)
input_embeddings.shape

torch.Size([1, 5, 16])

# Positional Encoding
RNN과 달리 DL은 병렬 처리하므로, 입력 데이터의 순서 정보 필요.

## Absolute Positional Encoding
sin, cos 활용 수식 통해 위치 정보 입력  
pros. 간단 구현 가능  
cons. token 간 상대적 위치 정보 활용 X  
cons. 학습 데이터에는 없는 긴 문장 추론 성능 떨어짐  

## Relative Positional Encoding
절대적 위치 인코딩의 단점 해결을 위해 만듦  
(자세한 설명은 나중에...)

In [5]:
embedding_dim = 16
max_position = 12 # 최대 토큰 수
embed_layer = nn.Embedding(len(str2idx), embedding_dim)
position_embed_layer = nn.Embedding(max_position, embedding_dim)

position_ids = torch.arange(len(input_ids), dtype=torch.long).unsqueeze(0)
position_encodings = position_embed_layer(position_ids)
token_embeddings = embed_layer(torch.tensor(input_ids))
token_embeddings = token_embeddings.unsqueeze(0)

input_embeddings = token_embeddings + position_encodings # 최종 입력 데이터(임베딩+포지션 인코딩)
input_embeddings.shape

torch.Size([1, 5, 16])

# Attention
단어 사이의 관계를 계산해서 관련 깊은 단어 구분한다.

## Query
검색할 단어(Embedding)
## Key
각 단어(Embedding)의 특징
## Value
Query와의 관련된(거리) 단어(Embedding)
  
## 과정
- 검색: Embeding,Wk(key weight) = k1...n - Wq(query weight),q1
- 결과(q1 재해석): Embeding,Wv(value weight) = v1...n - Wk,q1 의 가중합

In [6]:
head_dim = 16

weight_q = nn.Linear(embedding_dim, head_dim) # Query Weight
weight_k = nn.Linear(embedding_dim, head_dim) # Key Weight
weight_v = nn.Linear(embedding_dim, head_dim) # Value Weight

querys = weight_q(input_embeddings)
keys = weight_k(input_embeddings)
values = weight_v(input_embeddings) 

## 스케일 점곱
1. score = query * key / dim_k(임베딩 차원 수) 제곱근(분산 커짐 막기)
2. score를 합이 1이 되도록 softmax를 취해 가중치(weight)로 변환
3. 입력 동일 출력 형태 = weight * value  
(@ = 행렬 곱)

In [7]:
from math import sqrt
import torch.nn.functional as F

def compute_attention(querys, keys, values, is_causal=False):
    dim_k = querys.size(-1)
    scores = querys @ keys.transpose(-2, -1) / sqrt(dim_k)  # 1
    weights = F.softmax(scores, dim=-1)                     # 2
    return weights @ values                                 # 3

In [8]:
print("원본 입력 형태: ", input_embeddings.shape)

after_attention_embeddings = compute_attention(querys, keys, values)
print("어텐션 적용 후 형태: ", after_attention_embeddings.shape)

원본 입력 형태:  torch.Size([1, 5, 16])
어텐션 적용 후 형태:  torch.Size([1, 5, 16])


In [9]:
class AttentionHead(nn.Module):
    def __init__(self, token_embed_dim, head_dim, is_causal=False):
        super().__init__()
        self.is_causal = is_causal
        self.weight_q = nn.Linear(token_embed_dim, head_dim)
        self.weight_k = nn.Linear(token_embed_dim, head_dim)
        self.weight_v = nn.Linear(token_embed_dim, head_dim)

    def forward(self, querys, keys, values):
        outputs = compute_attention(
            self.weight_q(querys),
            self.weight_k(keys),
            self.weight_v(values),
            is_causal=self.is_causal
        )                           
        return outputs
    
attention_head = AttentionHead(embedding_dim, embedding_dim)
after_attention_embeddings = attention_head(input_embeddings, input_embeddings, input_embeddings)
print("원본 입력 형태: ", input_embeddings.shape)
print("어텐션 적용 후 형태: ", after_attention_embeddings.shape)

원본 입력 형태:  torch.Size([1, 5, 16])
어텐션 적용 후 형태:  torch.Size([1, 5, 16])


## Multy Head Attention
여러 Attention 연산을 동시에 적용 = 성능 높임

1. 헤드 수(n_head)만큼 연산 수행 위해 query, key, value를 n_head개로 나눈다(쪼갠다).
2. 각각의 attention 계산
3. 입력과 같은 형태로 다시 변환
4. 선형 층을 통과 시키고 최종 결과 반환

In [10]:
class MultiheadAttention(nn.Module):
    def __init__(self, token_embed_dim, d_model, n_head, is_causal=False):
        super().__init__()
        self.n_head = n_head
        self.is_causal = is_causal
        self.weight_q = nn.Linear(token_embed_dim, d_model)
        self.weight_k = nn.Linear(token_embed_dim, d_model)
        self.weight_v = nn.Linear(token_embed_dim, d_model)
        self.concat_linear = nn.Linear(d_model, d_model)

    def forward(self, querys, keys, values):
        B, T, C = querys.size()
        # 1
        querys = self.weight_q(querys).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        keys = self.weight_k(keys).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        values = self.weight_v(values).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        attention = compute_attention(querys, keys, values, self.is_causal) # 2
        output = attention.transpose(1, 2).contiguous().view(B, T, C)       # 3
        output = self.concat_linear(output)                                 # 4
        return output
    
n_head = 4
mh_attention = MultiheadAttention(embedding_dim, embedding_dim, n_head)
after_attention_embeddings = mh_attention(input_embeddings, input_embeddings, input_embeddings)
after_attention_embeddings.shape

torch.Size([1, 5, 16])

# Normalization
Deep Learning Model에서 Input Data의 분포를 일정하게 만듦  
(단위 차이가 나는 경우 결과값(학습)의 이상 발생 우려,  
특히 층이 깊어질 수록 분포의 차이가 많이 날 확률이 높음)  
- 안정된 학습
- 빠른 연산 속도

## Batch Normalization
Input Data를 mini batch(일정 크기)로 잘라서 평균/표준 편차 구함
- 자연어의 경우 Input Data의 크기가 다양하므로 효과 차이가 많이 난다.
- 주로 이미지 처리에 사용

## Layer Normalization
Token Embedding의 평균/표준 편차 구함
- post norm: 각각 Attention & Feed Forward 이후 정규화
- pre norm: 각각 Attention & Feed Forward 이전 정규화 (학습에 더 안정적)

In [11]:
norm = nn.LayerNorm(embedding_dim)
norm_x = norm(input_embeddings)
norm_x.shape

torch.Size([1, 5, 16])

In [12]:
norm_x.mean(dim=-1).data, norm_x.std(dim=-1).data # 실제 평균 & 표준 편차

(tensor([[-2.2352e-08,  2.2352e-08, -2.2352e-08,  4.4703e-08,  3.7253e-08]]),
 tensor([[1.0328, 1.0328, 1.0328, 1.0328, 1.0328]]))

## Feed Forward Layer
Input Text 전체를 이해하는 층  
Data의 특징을 학습 하는 완전 연결 층(Fully Connected Layer)  
- 선형(linear) 층, 드롭아웃(dropout) 층, 활성화 함수, 층 정규화(norm)으로 구성
- Embedding 차원(dim) 유지해야 층 쌓아 확장 가능 하기 때문에 차원 유지 필수  
(Input > dim 2-3배 늘림 > 연산 > dim 원상 복구 > Output)

In [13]:
class PreLayerNormFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.GELU()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src):
        x = self.norm(src)
        x = x + self.linear2(self.dropout1(self.activation(self.linear1(x))))
        x = self.dropout2(x)
        return x

# Encoder
Input > Tokenization > Embedding > Positional Encoding  
\> Layer Norm > Multi Head Attention + Residual Connection(잔차 연결)  
\> Layer Norm > Feed Forward Layer + Residual Connection(잔차 연결)  
\> Layer Norm  (Ne 번 반복)  

## 잔차 연결(Residual Connection)
안정적인 학습을 위해 Input을 그대로 다시 결과에 더해준다.

In [14]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.attn = MultiheadAttention(d_model, d_model, nhead)
        self.norm = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.feed_forward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout)

    def forward(self, src):
        norm_x = self.norm(src)
        attn_output = self.attn(norm_x, norm_x, norm_x)
        x = src + self.dropout1(attn_output) # residual connection

        x = self.feed_forward(x)
        return x

In [15]:
import copy

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers):
        super().__init__()
        self.layers = get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src):
        output = src
        for mod in self.layers:
            output = mod(output)
        return output

# Decoder
앞에서 생성한 Token 기반으로 다음 작성(순차적 = causal(인과적), self-regressive(자기 회귀적))  
but, Attention 그대로 사용 시 아직 작성 안 한 Token도 확인하는 문제 생김  
so, 이전 작성된 Token만 확인하는 **Masked Attention** 사용
- 마스크는 1 & -inf(infinite)로 된 행렬
- -inf 부분은 아직 생성 안 된 부분으로 softmax 시 가중치=0

In [16]:
def compute_attention(querys, keys, values, is_causal=False):
    dim_k = querys.size(-1)
    scores = querys @ keys.transpose(-2, -1) / sqrt(dim_k)
    if is_causal:
        query_length = querys.size(2)
        key_length = keys.size(2)
        temp_mask = torch.ones(query_length, key_length,
                               dtype=torch.bool).tril(diagonal=0)
        scores = scores.masked_fill(temp_mask == False, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ values

## Cross Attention
Decoder가 문장 생성(Query) 시 Encoder의 정보(결과값=Key)를 기반으로 도출(Encoder Value)

In [17]:
class TrnaformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, d_model, nhead)
        self.multyhead_attn = MultiheadAttention(d_model, d_model, nhead)
        self.feed_forward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, target, encoder_output, is_causal=True):
        # self attention
        x = self.norm1(target)
        x = x + self.dropout1(self.self_attn(x, x, x))

        # cross attention
        x = self.norm2(x)
        x = x + self.dropout2(self.multihead_attn(x, encoder_output, encoder_output, is_causal = is_causal))

        # feed forward
        x = self.feed_forward(x)
        return x

In [18]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers):
        super().__init__()
        self.layers = get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
    
    def forward(self, target, src):
        output = target
        for mod in self.layers:
            output = mod(target, src)
        return output

# 참고
- [DL의 layer(hidden-layer)](https://slowblooms.tistory.com/668)
- [RNN 깊을 경우 단점(gradient vanishing/exploding)](https://velog.io/@nayeon_p00/%EB%94%A5%EB%9F%AC%EB%8B%9D-%EA%B8%B0%EC%B4%88%EA%B0%9C%EB%85%90-%EA%B8%B0%EC%9A%B8%EA%B8%B0-%EC%86%8C%EC%8B%A4%EA%B3%BC-%ED%8F%AD%EC%A3%BC-Gradient-Vanishing-Exploding)
- [Attention](https://mz-moonzoo.tistory.com/106)
- [FNN Linear/Dropout layer](https://share.google/aimode/gOw5ZkMCG8JVuEwbr)
- [Cross Attention](https://pupbani.tistory.com/327)
- [Cross Attention-쉬운 설명 ver](https://velog.io/@hbcho/Cross-Attention-%EC%99%84%EB%B2%BD-%EC%9D%B4%ED%95%B4-%EC%98%A4%EB%A6%AC%EC%A7%80%EB%84%90-%ED%8A%B8%EB%9E%9C%EC%8A%A4%ED%8F%AC%EB%A8%B8%EB%B6%80%ED%84%B0-DiT%EC%9D%98-%EC%A1%B0%EA%B1%B4%EB%B6%80-%EC%9E%85%EB%A0%A5%EA%B9%8C%EC%A7%80)
